# Premotor neuronal activity

Run with `movement_bouts/` as the kernel's working directory.

`movement_bouts` is the reference table for all behavioral metadata. The neural
output has one `spike_counts` array per bout, indexed by the **same original bout
index**. Each array has shape `(neurons, 5 ms time bins)` and excludes movement onset.
Neuron IDs are stored once per session, indexed by `session_info.index` (`idx`).

See [the shared analysis format](ANALYSIS_DATA_FORMAT.md) for the data/labels
interface used by PCA, future UMAP analyses, and evaluation helpers in `evaluation.py`.


# 1. Load the reference tables and define parameters


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

from utils import (
    extract_all_premotor_bouts, prepare_pca_input,
    fit_pca_by_condition, plot_pca_session,
)

data_dir = Path("../../data").resolve()
results_dir = data_dir / "derived" / "movement_bouts"
premotor_dir = data_dir / "derived" / "premotor_activity"

movement_bouts = pd.read_pickle(results_dir / "movement_bouts.pkl")
session_info = pd.read_pickle(results_dir / "session_info.pkl")
parameters = json.loads((results_dir / "parameters.json").read_text(encoding="utf-8"))

print(f"Loaded {len(movement_bouts)} movement bouts across {len(session_info)} session entries.")


In [ ]:
DT = parameters["DT"]
PREMOTOR_DURATION_S = 0.5  # Window: [-0.5, 0) seconds relative to movement onset.
N_PREMOTOR_BINS = round(PREMOTOR_DURATION_S / DT)
N_BINS = 5  # Averaging windows: 5 = 100 ms, 2 = 250 ms, 1 = 500 ms.


# 2. Extract and save the original-resolution counts

The helper loads each source collection once, then extracts session by session.
It skips bouts without a complete window and records their indices and reasons.
Behavioral metadata stays in `movement_bouts`; its indices are never reset.

To extract only one session, pass `session_info.loc[[idx]]` instead of `session_info`.


In [ ]:
neuronal_bouts, neuron_ids_by_session, excluded_bouts = extract_all_premotor_bouts(
    data_dir, movement_bouts, session_info,
    dt=DT, premotor_duration_s=PREMOTOR_DURATION_S,
)

print(f"Extracted {len(neuronal_bouts)} bouts; skipped {len(excluded_bouts)}.")
if len(excluded_bouts):
    display(excluded_bouts.head())


In [ ]:
import hashlib
from datetime import datetime, timezone

premotor_dir.mkdir(parents=True, exist_ok=True)
neuronal_bouts.to_pickle(premotor_dir / "neuronal_bouts.pkl")
neuron_ids_by_session.to_pickle(premotor_dir / "neuron_ids_by_session.pkl")
excluded_bouts.to_pickle(premotor_dir / "excluded_bouts.pkl")

# Fingerprints link bout indices to the exact reference tables used here.
extraction_parameters = {
    "DT": DT,
    "PREMOTOR_DURATION_S": PREMOTOR_DURATION_S,
    "N_PREMOTOR_BINS": N_PREMOTOR_BINS,
    "window_s": [-PREMOTOR_DURATION_S, 0],
    "onset_included": False,
    "units": "spike counts per original bin",
    "session_indices": [int(idx) for idx in neuron_ids_by_session.index],
    "source_directory": str(results_dir),
    "source_sha256": {
        name: hashlib.sha256((results_dir / name).read_bytes()).hexdigest()
        for name in ("movement_bouts.pkl", "session_info.pkl")
    },
    "bout_detection_parameters": parameters,
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
}
(premotor_dir / "parameters.json").write_text(
    json.dumps(extraction_parameters, indent=2) + "\n", encoding="utf-8"
)
print(f"Saved premotor tables to {premotor_dir}")


# 3. Prepare one session for PCA

If starting from the saved extraction, reload these two tables instead of
rerunning section 2. Run section 1 for the reference tables and parameters.

```python
neuronal_bouts = pd.read_pickle(premotor_dir / "neuronal_bouts.pkl")
neuron_ids_by_session = pd.read_pickle(premotor_dir / "neuron_ids_by_session.pkl")
```

`prepare_pca_input()` selects the session, averages counts into `N_BINS` equal
windows, converts to spikes/s, and creates one observation per bout/window.
`pca_input` contains only neural firing rates; `pca_labels` repeats all original
bout metadata and adds window labels. Both use `(bout_id, bin_start_s)` as their
index. Bout-level speed/direction labels are not recalculated per window.

Changing `idx` or `N_BINS` only requires rerunning this call and the cells below.
The helper keeps preparation in memory and does not fit PCA or save new files.


In [ ]:
idx = 0
pca_input, pca_labels = prepare_pca_input(
    neuronal_bouts, movement_bouts, neuron_ids_by_session,
    idx=idx, dt=DT, n_bins=N_BINS,
)

print(f"idx={idx}: PCA input {pca_input.shape} (bout-window observations, neurons)")
display(pca_input.head(N_BINS))
display(pca_labels.head(N_BINS))


# 4. Inspect one bout: all neurons and averaging windows


In [ ]:
import matplotlib.pyplot as plt

# Choose one retained bout by its position within the prepared session.
bout_number = 0
bout_id = pca_input.index.get_level_values("bout_id").unique()[bout_number]
bout_rates = pca_input.xs(bout_id, level="bout_id")
bout_labels = pca_labels.xs(bout_id, level="bout_id")
condition = bout_labels["condition"].iloc[0]
condition_name = {0: "Tracking", 1: "Playback"}.get(condition, str(condition))
cmap = {0: "Reds", 1: "Greys"}.get(condition, "viridis")
edges_ms = np.append(bout_rates.index.to_numpy(), bout_labels["bin_end_s"].iloc[-1]) * 1000
window_ms = edges_ms[1] - edges_ms[0]

fig, ax = plt.subplots(figsize=(11, max(4, len(bout_rates.columns) * 0.3)))
heatmap = ax.pcolormesh(
    edges_ms, np.arange(len(bout_rates.columns) + 1) - 0.5,
    bout_rates.to_numpy().T, cmap=cmap, vmin=0, shading="flat",
)
ax.set(
    yticks=np.arange(len(bout_rates.columns)), yticklabels=bout_rates.columns,
    xlabel="Time relative to movement onset (ms)", ylabel="Neuron",
    title=f"idx={idx} | bout ID {bout_id} | {condition_name} | {window_ms:g} ms windows",
    xlim=(edges_ms[0], edges_ms[-1]),
)
ax.invert_yaxis()
fig.colorbar(heatmap, ax=ax, label="Firing rate (spikes/s)")
fig.tight_layout()
plt.show()


# 5. Fit separate PCAs for tracking and playback

`fit_pca_by_condition()` fits the two conditions independently, retaining all
components without variance standardization or whitening. `pca_fits[0]` is
tracking and `pca_fits[1]` is playback. Each entry contains a fitted `model` and
`results` (PCA coordinates joined to the original bout/window labels).


In [ ]:
pca_fits = fit_pca_by_condition(pca_input, pca_labels)
for condition, fit in pca_fits.items():
    name = {0: "Tracking", 1: "Playback"}[condition]
    print(f"{name}, idx={idx}: {len(fit['results'])} observations")
    display(pd.DataFrame(
        {"explained_variance_ratio": fit["model"].explained_variance_ratio_},
        index=[f"PC{i + 1}" for i in range(fit["model"].n_components_)],
    ))


# 6. Plot each PCA in 3D by direction and starting position

Crosses mark leftward movement (-1), circles rightward movement (+1).
Squares mark zero/unknown initial direction, if present. Color represents the
bout starting position, using the same color scale for both separate PCAs.
Every point is a bout/window observation; direction and position are bout-level labels.

The three axes are PC1, PC2, and PC3 from each condition-specific fit.


In [ ]:
import matplotlib.pyplot as plt

fig_pca, axes_pca = plot_pca_session(pca_fits, title=f"idx={idx}: premotor activity")
plt.show()


# 7. Compute and save every session's PCAs (run once per averaging choice)

Load the saved premotor counts, prepare each session, and fit tracking/playback
separately using the same helpers as above. `pca_cache["sessions"][idx][condition]`
holds a `model` and a `results` table. Models retain neuron names, weights,
centering values and explained variance; result tables retain coordinates and
all bout/window labels. No original recording files are loaded.

Save both together in `pca/pca_5bins.pkl` (the filename follows `N_BINS`), along
with the settings, session identities, and fixed starting-position color limits.
To browse an existing file, skip this section and run sections 1 and 8.


In [ ]:
from datetime import datetime, timezone
import sklearn

neuronal_bouts = pd.read_pickle(premotor_dir / "neuronal_bouts.pkl")
neuron_ids_by_session = pd.read_pickle(premotor_dir / "neuron_ids_by_session.pkl")
pca_sessions = {}
skipped_sessions = {}

for session_idx in session_info.index:
    bout_ids = movement_bouts.index[movement_bouts["idx"] == session_idx]
    if not bout_ids.isin(neuronal_bouts.index).any():
        skipped_sessions[session_idx] = "No extracted bouts"
        continue
    session_input, session_labels = prepare_pca_input(
        neuronal_bouts, movement_bouts, neuron_ids_by_session,
        idx=session_idx, dt=DT, n_bins=N_BINS,
    )
    fits = fit_pca_by_condition(session_input, session_labels)
    if fits:
        pca_sessions[session_idx] = fits
    else:
        skipped_sessions[session_idx] = "Fewer than two observations per condition"

pca_cache = {
    "settings": {
        "DT": DT, "N_BINS": N_BINS,
        "standardized": False, "whiten": False,
        "sklearn_version": sklearn.__version__,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "extraction_parameters": json.loads((premotor_dir / "parameters.json").read_text(encoding="utf-8")),
    },
    "session_info": session_info.loc[list(pca_sessions)].copy(),
    "sessions": pca_sessions,
    "skipped_sessions": skipped_sessions,
    "position_limits": (
        float(movement_bouts["initial_position"].min()),
        float(movement_bouts["initial_position"].max()),
    ),
}
pca_dir = premotor_dir / "pca"
pca_dir.mkdir(exist_ok=True)
pca_path = pca_dir / f"pca_{N_BINS}bins.pkl"
pd.to_pickle(pca_cache, pca_path)
print(f"Saved {sum(len(fits) for fits in pca_sessions.values())} PCAs across {len(pca_sessions)} sessions to {pca_path}")
if skipped_sessions:
    print("Skipped sessions:", skipped_sessions)


# 8. Browse PCA by starting position

This viewer has its own session slider. All points are dots, colored by starting
position with fixed limits across sessions. Tracking and playback cameras are
linked within this viewer. Drag to rotate, scroll to zoom, and hover for labels.

Section 9 shows the same saved PCA coordinates colored by direction, with a
separate slider and independent camera controls. No refitting occurs in either.


In [ ]:
from importlib import reload
import utils

reload(utils)
from utils import make_interactive_pca_browser

# Close the former combined viewer and any previous position viewer on rerun.
for browser_name in ("pca_browser", "position_browser"):
    if browser_name in globals():
        old_browser = globals()[browser_name]
        for child in old_browser.children:
            child.close()
        old_browser.close()

pca_cache = pd.read_pickle(premotor_dir / "pca" / f"pca_{N_BINS}bins.pkl")
position_browser = make_interactive_pca_browser(pca_cache, color_by="position", width=1400)
display(position_browser)


# 9. Browse direction-colored PCA, filtered by starting position

This viewer has independent session and position controls. Select all bouts,
starting left of the midpoint (`initial_position < 14`), or starting at/right of
it (`>= 14`). The midpoint is half of the reconstructed 0-28 position scale;
adjust `midpoint` below if needed. All windows of a bout share its filter.

Dots remain blue for leftward movement and red for rightward movement. Filtering
only hides points in the existing PCA: axes retain their full-session ranges,
models are not refitted, and the two cameras remain linked within this viewer.


In [ ]:
from importlib import reload
import utils

make_interactive_pca_browser = reload(utils).make_interactive_pca_browser
# Close either former direction viewer when this cell is rerun.
for browser_name in ("direction_browser", "filtered_direction_browser"):
    if browser_name in globals():
        old_browser = globals()[browser_name]
        for child in old_browser.children:
            child.close()
        old_browser.close()

direction_browser = make_interactive_pca_browser(
    pca_cache, color_by="direction", filter_position=True, midpoint=14.0, width=1400,
)
display(direction_browser)


# 10. Inspect one bout's premotor trajectory

This independent viewer has a session slider and a bout slider. The selected
bout's windows are labeled by bin number, colored light to dark with time, and
connected in chronological order. Hover gives each window's start and end time.
Other bouts are faint and colored by starting position, using the same fixed
Viridis scale as the position viewer. The selected trajectory keeps its separate
light-to-dark time colors. A bout belongs to one condition, so its highlighted
trajectory appears only in the corresponding tracking or playback panel.

Both cameras stay linked within this viewer. Axis ranges stay fixed for the
session when switching bouts. These are the saved first three PC coordinates;
no PCA is refitted and no new data are saved.


In [ ]:
from importlib import reload
import utils

make_bout_pca_browser = reload(utils).make_bout_pca_browser
if "bout_browser" in globals():
    for child in bout_browser.children:
        child.close()
    bout_browser.close()

bout_browser = make_bout_pca_browser(pca_cache, width=1400)
display(bout_browser)


# 11. Project playback activity into tracking PCA space

Reuse each session's saved **tracking** PCA for both conditions. Playback rates
are transformed with the tracking neuron's mean and loading vectors; they are
not recentered by the playback mean, and no PCA is fitted again. The existing
separate-PCAs results remain unchanged.

Run the computation cell once to save projections for all sessions. On later
visits, run the viewer cell only. Both panels use matching axis limits and linked
cameras: tracking on the left, playback in tracking axes on the right. Color is
starting position. Percentages on both panels describe **tracking** variance,
not the fraction of playback variance explained by those axes.


In [ ]:
from importlib import reload
import utils

project_onto_tracking_pca = reload(utils).project_onto_tracking_pca
reference_pca_path = premotor_dir / "pca" / f"pca_{N_BINS}bins.pkl"
reference_pca_cache = pd.read_pickle(reference_pca_path)
neuronal_bouts = pd.read_pickle(premotor_dir / "neuronal_bouts.pkl")
neuron_ids_by_session = pd.read_pickle(premotor_dir / "neuron_ids_by_session.pkl")
tracking_projection_sessions = {}

for session_idx, fits in reference_pca_cache["sessions"].items():
    if 0 not in fits:
        continue
    session_input, session_labels = utils.prepare_pca_input(
        neuronal_bouts, movement_bouts, neuron_ids_by_session,
        idx=session_idx, dt=reference_pca_cache["settings"]["DT"],
        n_bins=reference_pca_cache["settings"]["N_BINS"],
    )
    tracking_projection_sessions[session_idx] = project_onto_tracking_pca(
        session_input, session_labels, fits[0]["model"],
    )

tracking_projection_cache = {
    "projection_basis": "tracking",
    "settings": {**reference_pca_cache["settings"], "source_pca_file": reference_pca_path.name},
    "session_info": reference_pca_cache["session_info"].loc[list(tracking_projection_sessions)],
    "position_limits": reference_pca_cache["position_limits"],
    "sessions": tracking_projection_sessions,
}
tracking_projection_path = premotor_dir / "pca" / f"pca_tracking_projection_{N_BINS}bins.pkl"
pd.to_pickle(tracking_projection_cache, tracking_projection_path)
print(f"Saved tracking-space projections for {len(tracking_projection_sessions)} sessions to {tracking_projection_path}")


In [ ]:
from importlib import reload
import utils

make_interactive_pca_browser = reload(utils).make_interactive_pca_browser
if "tracking_projection_browser" in globals():
    for child in tracking_projection_browser.children:
        child.close()
    tracking_projection_browser.close()

tracking_projection_cache = pd.read_pickle(
    premotor_dir / "pca" / f"pca_tracking_projection_{N_BINS}bins.pkl"
)
tracking_projection_browser = make_interactive_pca_browser(
    tracking_projection_cache, color_by="position", width=1400,
)
display(tracking_projection_browser)


# 12. Browse PCA by destination position

Color each premotor observation by its bout's saved `final_position` (the position
at the movement endpoint). This is an observed destination, not an inferred target.
All windows of a bout share the same destination label. Unfinished bouts with no
known endpoint are gray. The positional scale matches the starting-position view.

This independent session slider uses the existing separate tracking/playback PCAs,
with linked cameras. Only the coloring changes; no extraction or fitting is rerun.


In [ ]:
from importlib import reload
import utils

make_interactive_pca_browser = reload(utils).make_interactive_pca_browser
if "destination_browser" in globals():
    for child in destination_browser.children:
        child.close()
    destination_browser.close()

destination_browser = make_interactive_pca_browser(
    pca_cache, color_by="destination", width=1400,
)
display(destination_browser)


# 13. How closely does destination follow starting position?

Each point is one completed bout from `movement_bouts`, not one premotor window.
Tracking and playback plots compare starting and final positions; the dashed
identity line marks identical endpoints. Titles show Pearson correlation and
median absolute net displacement, in the saved position units.

The third panel is the cumulative distribution of `abs(final_position -
initial_position)`: its height at any distance is the fraction of bouts whose
endpoint is that close to the start. This avoids choosing an arbitrary threshold
for "close". The slider selects a session, matching our session-specific PCAs.
Unknown endpoints are excluded and counted.

Small net displacement can mean a short movement or a return to the starting
position after a longer path. It is not the same as `total_movement`. These are
descriptive associations; they do not isolate destination coding from start coding.


In [ ]:
from importlib import reload
import utils

make_start_end_position_browser = reload(utils).make_start_end_position_browser
if "start_end_browser" in globals():
    for child in start_end_browser.children:
        child.close()
    start_end_browser.close()

start_end_browser = make_start_end_position_browser(movement_bouts, session_info, width=1500)
display(start_end_browser)


# 14. Destination-colored PCA excluding bouts ending near their start

Adjust the minimum absolute net displacement `abs(final_position - initial_position)`.
The initial cutoff is 2 position units: keep bouts with displacement >= 2 and hide
all windows of bouts below that cutoff. A cutoff d corresponds to perpendicular
distance d/sqrt(2) from the identity line in section 13. Set zero to show all bouts
with known endpoints; unknown endpoints are excluded at every cutoff.

This viewer reuses the existing separate PCAs, colored by destination. The slider
only filters displayed points; PCA models, full-session axis ranges, colors and
linked cameras remain unchanged. Panel headings show retained/total observations.


In [ ]:
from importlib import reload
import utils

make_interactive_pca_browser = reload(utils).make_interactive_pca_browser
if "displacement_browser" in globals():
    for child in displacement_browser.children:
        child.close()
    displacement_browser.close()

displacement_browser = make_interactive_pca_browser(
    pca_cache, color_by="destination", filter_displacement=True,
    min_displacement=2.0, width=1400,
)
display(displacement_browser)
